# 04 — Modeling on Cleaned Data

This notebook builds a defensible baseline and logistic regression workflow for predicting 30-day heart failure readmission.

It uses `data/processed/cleaned.csv`, which is produced by `01_cleaning.ipynb`. Missing values are imputed inside the modeling pipeline after the train/test split to avoid data leakage.

Modeling steps:
- load the cleaned dataset
- separate predictors and target
- create a stratified train/test split
- build preprocessing pipelines for numeric and categorical variables
- compare a dummy baseline with logistic regression
- evaluate accuracy, specificity, precision, recall, F1, ROC-AUC, and average precision
- save model comparison tables and figures
- interpret logistic regression coefficients as odds ratios


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from src.config import PROCESSED_DIR, TARGET, FIGURES_DIR, TABLES_DIR
except ModuleNotFoundError:
    # Fallback for running the notebook outside the project package structure.
    PROJECT_DIR = Path.cwd().parent
    PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
    FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'
    TABLES_DIR = PROJECT_DIR / 'reports' / 'tables'
    TARGET = 'readmitted_30d'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)


## Load cleaned data

This notebook should be run after `01_cleaning.ipynb`, which creates `cleaned.csv`.

In [ ]:
cleaned_path = PROCESSED_DIR / 'cleaned.csv'

if not cleaned_path.exists():
    raise FileNotFoundError(
        f'{cleaned_path} was not found. Run 01_cleaning.ipynb first to create cleaned.csv.'
    )

df = pd.read_csv(cleaned_path)

print('Rows, columns:', df.shape)
df.head()

In [ ]:
# Confirm target balance before modeling.
target_balance = (
    df[TARGET]
    .value_counts(normalize=True)
    .rename('proportion')
    .reset_index()
    .rename(columns={'index': TARGET})
)

target_balance.to_csv(TABLES_DIR / 'tbl_model_target_balance.csv', index=False)
target_balance

## Define predictors and target

`patient_id` should already be removed by cleaning. If it is still present, it is dropped here as a safety check because identifiers should not be used as predictors.

In [ ]:
if 'patient_id' in df.columns:
    df = df.drop(columns=['patient_id'])

X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## Train/test split

A stratified split keeps the readmission rate similar in the training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'rows': [len(y_train), len(y_test)],
    'readmission_rate': [y_train.mean(), y_test.mean()],
})

split_summary.to_csv(TABLES_DIR / 'tbl_train_test_split.csv', index=False)
split_summary

## Build preprocessing pipeline

Numeric variables are median-imputed and scaled. Categorical variables are imputed with the most frequent category and one-hot encoded. These steps occur inside the pipeline so preprocessing is learned from the training data only.

In [ ]:
# Use sparse_output for newer scikit-learn versions and sparse for older versions.
try:
    encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse=False)

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

## Define models

The dummy model is the baseline. Logistic regression is the main interpretable model for this project.

In [ ]:
baseline_model = DummyClassifier(strategy='most_frequent')

logistic_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000, solver='liblinear')),
    ]
)

## Evaluation helper

Because this is a readmission-risk problem, recall and precision are especially important alongside specificity and ROC-AUC. Recall tells us how many readmitted patients the model catches. Specificity tells us how many non-readmitted patients the model correctly avoids flagging. Precision tells us how many predicted readmissions were actually readmissions.


In [ ]:
def calculate_specificity(y_true, y_pred):
    """Return specificity, also called the true negative rate."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp) if (tn + fp) else np.nan


def evaluate_classifier(name, model, X_train, X_test, y_train, y_test):
    """Fit a classifier and return standard classification metrics."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, 'decision_function'):
        y_score = model.decision_function(X_test)
    else:
        y_score = y_pred

    return {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'specificity': calculate_specificity(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_score),
        'average_precision': average_precision_score(y_test, y_score),
    }


In [ ]:
results = [
    evaluate_classifier('Dummy baseline', baseline_model, X_train, X_test, y_train, y_test),
    evaluate_classifier('Logistic regression', logistic_model, X_train, X_test, y_train, y_test),
]

model_results = pd.DataFrame(results).sort_values('roc_auc', ascending=False)
model_results.to_csv(TABLES_DIR / 'tbl_model_results.csv', index=False)
model_results.round(4)

## Cross-validation on training data

Cross-validation estimates how stable logistic regression performance is before final test-set evaluation.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

specificity_scorer = make_scorer(calculate_specificity)

cv_scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        'accuracy': 'accuracy',
        'specificity': specificity_scorer,
        'precision': 'precision',
        'recall': 'recall',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'average_precision': 'average_precision',
    },
    n_jobs=-1,
)

cv_summary = pd.DataFrame({
    metric.replace('test_', ''): [values.mean(), values.std()]
    for metric, values in cv_scores.items()
    if metric.startswith('test_')
}, index=['mean', 'std']).T.reset_index().rename(columns={'index': 'metric'})

cv_summary.to_csv(TABLES_DIR / 'tbl_logistic_cv_results.csv', index=False)
cv_summary.round(4)


## Final logistic regression evaluation

Fit logistic regression on the training data and evaluate it on the held-out test set.

In [ ]:
logistic_model.fit(X_train, y_train)

y_pred = logistic_model.predict(X_test)
y_proba = logistic_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cmd = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not readmitted', 'Readmitted'],
)
cmd.plot(values_format='d')
plt.title('Logistic Regression Confusion Matrix')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_logistic_confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title('Logistic Regression ROC Curve')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_logistic_roc_curve.png', dpi=300)
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

plt.figure()
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Logistic Regression Precision-Recall Curve')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_logistic_precision_recall_curve.png', dpi=300)
plt.show()

## Validation threshold review

The default classification threshold is 0.50. For readmission-risk screening, a lower threshold may increase recall, but it can also create more false positives.

To avoid tuning thresholds directly on the held-out test set, this section creates an additional validation split from the training data. The validation table is used only to illustrate the recall/precision tradeoff and identify candidate thresholds. Final test-set metrics are reported separately for the default threshold and the selected screening threshold.

In [ ]:
THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)

threshold_model = clone(logistic_model)
threshold_model.fit(X_fit, y_fit)
val_proba = threshold_model.predict_proba(X_val)[:, 1]

threshold_rows = []

for threshold in THRESHOLDS:
    preds = (val_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    threshold_rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_val, preds),
        'specificity': specificity,
        'precision': precision_score(y_val, preds, zero_division=0),
        'recall': recall_score(y_val, preds, zero_division=0),
        'f1': f1_score(y_val, preds, zero_division=0),
        'predicted_positive_rate': preds.mean(),
    })

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.to_csv(TABLES_DIR / 'tbl_logistic_threshold_review_validation.csv', index=False)
threshold_results.round(4)

## Test metrics for default and screening thresholds

The held-out test set is not used for the full threshold sweep. It is used here only to report final performance for the default 0.50 threshold and the selected 0.40 screening threshold.

In [ ]:
selected_thresholds = [0.50, 0.40]
test_threshold_rows = []

for threshold in selected_thresholds:
    preds = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    test_threshold_rows.append({
        'threshold': threshold,
        'accuracy': accuracy_score(y_test, preds),
        'specificity': specificity,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'true_negatives': tn,
        'false_positives': fp,
        'false_negatives': fn,
        'true_positives': tp,
    })

test_threshold_results = pd.DataFrame(test_threshold_rows)
test_threshold_results.to_csv(TABLES_DIR / 'tbl_logistic_selected_threshold_test_metrics.csv', index=False)
test_threshold_results.round(4)

## Logistic Regression report row

This table surfaces the Logistic Regression values needed for the team's final model comparison table, including specificity. The row uses the selected 0.40 screening threshold for precision, recall, specificity, and F1, while accuracy and AUROC come from the default model evaluation.


In [ ]:
screening_threshold = 0.40
screening_metrics = test_threshold_results.loc[
    test_threshold_results['threshold'].eq(screening_threshold)
].iloc[0]

logistic_base_metrics = model_results.loc[
    model_results['model'].eq('Logistic regression')
].iloc[0]

baseline_accuracy = model_results.loc[
    model_results['model'].eq('Dummy baseline'), 'accuracy'
].iloc[0]

logistic_report_row = pd.DataFrame([{
    'model': 'Logistic Regression',
    'baseline_accuracy': baseline_accuracy,
    'test_accuracy': logistic_base_metrics['accuracy'],
    'specificity_at_0_40': screening_metrics['specificity'],
    'precision_at_0_40': screening_metrics['precision'],
    'recall_at_0_40': screening_metrics['recall'],
    'f1_at_0_40': screening_metrics['f1'],
    'roc_auc': logistic_base_metrics['roc_auc'],
    'notes': 'Interpretable model; specificity, precision, recall, and F1 reported at selected 0.40 screening threshold.',
}])

logistic_report_row.to_csv(TABLES_DIR / 'tbl_logistic_report_comparison_row.csv', index=False)
logistic_report_row.round(4)


## Interpret coefficients as odds ratios

For logistic regression, a positive coefficient increases the predicted log-odds of readmission. Exponentiating the coefficient gives an odds ratio. Odds ratios above 1 indicate higher odds of readmission, while values below 1 indicate lower odds.

In [ ]:
def get_feature_names(preprocessor):
    """Get output feature names from the fitted ColumnTransformer."""
    feature_names = []

    if numeric_features:
        feature_names.extend(numeric_features)

    if categorical_features:
        cat_pipeline = preprocessor.named_transformers_['cat']
        encoder_step = cat_pipeline.named_steps['encoder']
        cat_names = encoder_step.get_feature_names_out(categorical_features).tolist()
        feature_names.extend(cat_names)

    return feature_names

fitted_preprocessor = logistic_model.named_steps['preprocessor']
feature_names = get_feature_names(fitted_preprocessor)
coefs = logistic_model.named_steps['model'].coef_[0]

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'odds_ratio': np.exp(coefs),
    'abs_coefficient': np.abs(coefs),
}).sort_values('abs_coefficient', ascending=False)

coef_table.to_csv(TABLES_DIR / 'tbl_logistic_coefficients.csv', index=False)
coef_table.head(15).round(4)

In [ ]:
top_coef = coef_table.head(15).sort_values('coefficient')

plt.figure(figsize=(8, 6))
plt.barh(top_coef['feature'], top_coef['coefficient'])
plt.xlabel('Logistic regression coefficient')
plt.title('Top Logistic Regression Coefficients')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_logistic_top_coefficients.png', dpi=300)
plt.show()

## Modeling notes for the technical report

Use this section to summarize the model findings after rerunning the notebook.

In [ ]:
summary_notes = pd.DataFrame({
    'topic': [
        'Data source',
        'Preprocessing',
        'Baseline',
        'Model',
        'Metrics',
        'Interpretation',
        'Threshold review',
        'Specificity',
        'Limitation',
    ],
    'note': [
        'Modeling used data/processed/cleaned.csv produced by the cleaning notebook.',
        'Numeric features were median-imputed and scaled; categorical features were mode-imputed and one-hot encoded inside the pipeline.',
        'A most-frequent dummy classifier was used as the baseline.',
        'Logistic regression was used as the main interpretable classification model.',
        'Performance was evaluated with accuracy, specificity, precision, recall, F1, ROC-AUC, and average precision.',
        'Coefficients were converted to odds ratios for practical interpretation.',
        'Threshold review used a validation split from the training data to avoid tuning on the held-out test set.',
        'Specificity is calculated as TN / (TN + FP) and is included in the model results and final report comparison row.',
        'The model estimates associations and risk patterns, but does not prove causation.',
    ],
})

summary_notes.to_csv(TABLES_DIR / 'tbl_modeling_report_notes.csv', index=False)
summary_notes
